# LoRA fine-tune of ESM-2 on BacDive phenotype prediction (Kaggle P100)

Trains the `PhenoLoRAModel` (ESM-2 t12 + LoRA r=8 + per-category mean-pool + 4 multi-task heads) for one group-K-fold split.

Pre-attached datasets:
- `miyuiu/bacdive-marker-sequences` → `/kaggle/input/bacdive-marker-sequences/marker_sequences.jsonl`
- `miyuiu/bacdive-tables` → `/kaggle/input/bacdive-tables/{bacdive_phenotypes,strain_catalog}.parquet`
- `miyuiu/microbe-model-code` → `/kaggle/input/microbe-model-code/` (auto-extracted by Kaggle; rebuilt into a package below)

In [ ]:
# Force-downgrade the full ML stack to a P100-compatible, internally-consistent set.
# - torch 2.4.1 / torchvision 0.19.1 / triton 3.0.0  : have sm_60 wheels for P100
# - transformers 4.46.0 / tokenizers 0.20.3 / accelerate 1.0.0 / peft 0.13.0
#   : last stable combo before transformers 5.x bumped torch + tokenizer ABIs.
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--force-reinstall', '--no-deps',
    '--index-url', 'https://download.pytorch.org/whl/cu121',
    'torch==2.4.1', 'torchvision==0.19.1',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--force-reinstall', '--no-deps',
    'transformers==4.46.0', 'tokenizers==0.20.3',
    'peft==0.13.0', 'accelerate==1.0.0', 'huggingface_hub==0.26.2',
])
subprocess.check_call([sys.executable, '-c', '''
import torch, transformers, peft, tokenizers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("peft:", peft.__version__)
print("CUDA arch list:", torch.cuda.get_arch_list() if torch.cuda.is_available() else "no GPU")
'''])

In [ ]:
# 0. Debug: show what's actually mounted from the attached datasets.
import os
print('contents of /kaggle/input/:')
for p in sorted(os.listdir('/kaggle/input')):
    print(' ', p)
    try:
        for c in sorted(os.listdir(f'/kaggle/input/{p}'))[:8]:
            print('   ', c)
    except Exception as e:
        print('    (cannot list:', e, ')')

In [ ]:
# Kaggle mounts owner-scoped private datasets at /kaggle/input/datasets/<owner>/<slug>/
# Find them dynamically so this works regardless of mount path.
import shutil, sys, glob
from pathlib import Path

def find_mount(slug):
    for cand in (f'/kaggle/input/{slug}', f'/kaggle/input/datasets/miyuiu/{slug}'):
        if Path(cand).exists():
            return Path(cand)
    matches = glob.glob(f'/kaggle/input/**/{slug}', recursive=True)
    return Path(matches[0]) if matches else None

src = find_mount('microbe-model-code')
assert src is not None, 'microbe-model-code dataset not mounted'
dst = Path('/kaggle/working/microbe_model')
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
sys.path.insert(0, '/kaggle/working')
print('package rebuilt at', dst)
print('contents:', sorted(p.name for p in dst.iterdir()))

# Cache the marker-seq + tables mount paths for later cells
MARKER_DIR = find_mount('bacdive-marker-sequences')
TABLES_DIR = find_mount('bacdive-tables')
print('MARKER_DIR =', MARKER_DIR)
print('TABLES_DIR =', TABLES_DIR)

In [ ]:
# Quick GPU + torch diagnostic before training.
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('compute capability:', torch.cuda.get_device_capability(0))
    print('torch CUDA arch list:', torch.cuda.get_arch_list())
    try:
        x = torch.randn(2, 2, device='cuda')
        y = x @ x.T
        print('matmul OK:', y.shape, y.dtype, y.sum().item())
    except Exception as e:
        print('matmul FAILED:', type(e).__name__, e)
    try:
        x = torch.randn(2, 2, device='cuda', dtype=torch.bfloat16)
        y = x @ x.T
        print('bf16 matmul OK:', y.shape, y.dtype)
    except Exception as e:
        print('bf16 matmul FAILED:', type(e).__name__, e)

In [ ]:
# Locate inputs (paths discovered in cell 2) and verify everything is reachable.
MARKER_SEQ = MARKER_DIR / 'marker_sequences.jsonl'
PHENOTYPES = TABLES_DIR / 'bacdive_phenotypes.parquet'
CATALOG    = TABLES_DIR / 'strain_catalog.parquet'

for p in (MARKER_SEQ, PHENOTYPES, CATALOG):
    assert p.exists(), f'Missing {p}'

from microbe_model.train.lora_model import LoraModelConfig
from microbe_model.train.lora_trainer import TrainConfig, train_lora
print('imports OK')

In [ ]:
# 4. Configure and launch the run.
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

FOLD = 0
EPOCHS = 1                       # one epoch fits comfortably in the 12-h session
ESM_MODEL = "facebook/esm2_t12_35M_UR50D"

model_cfg = LoraModelConfig(
    esm_model_name=ESM_MODEL,
    lora_r=8,
    gradient_checkpointing=True,
    max_seq_len=512,
    max_proteins_per_cat=6,
)
train_cfg = TrainConfig(
    fold=FOLD,
    epochs=EPOCHS,
    batch_size=2,
    grad_accum=8,
    lora_lr=1e-4,
    head_lr=1e-3,
    save_dir="/kaggle/working",
)

results = train_lora(
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    sequences_path=MARKER_SEQ,
    phenotypes_path=PHENOTYPES,
    catalog_path=CATALOG,
    device=device,
)

print("\n=== best epoch ===")
print(results["best"])

In [ ]:
# 5. Inspect outputs. /kaggle/working/ files are downloadable from the Output tab.
!ls -lh /kaggle/working/